# BanglaLLM 7B Instruct Few-Shot Government QA Evaluation — 3-Shot

This notebook evaluates **`BanglaLLM/bangla-llama-7b-instruct-v0.1`** in a reproducible **3-shot** setting on:

`/kaggle/input/datasets/akra1234/government/merged_test_data.csv`

Few-shot demonstrations are selected only from:

`/kaggle/input/datasets/akra1234/government/government_chat_train.jsonl`

## Experimental design

- Uses **3 fixed demonstrations** for every test question.
- Demonstrations are taken from the **training split only**.
- Exact train/test question overlap is removed before shot selection.
- Uses the same model, 4-bit NF4 quantization, generation cap, and evaluation metrics as the BanglaLLM zero-shot notebook.
- The only intended experimental difference from zero-shot is the presence of the demonstrations.
- Uses BanglaLLM's native Alpaca-style format:
  - `### Instruction:`
  - optional `### Input:`
  - `### Response:`
- Uses seeded sampling:
  - `do_sample=True`
  - `temperature=0.6`
  - `top_p=0.9`
  - `top_k=50`
- Uses `MAX_NEW_TOKENS=256` as the same safety cap used in the BanglaLLM zero-shot run.
- Does **not** retry 256 → 512 → 1024 → 2048 → 4096.
- Token-limit outputs are retained and evaluated; the token-limit rate is reported separately.
- Saves the exact demonstration rows so the few-shot prompt is auditable and reproducible.

## Important fairness option

For the strictest Qwen-vs-BanglaLLM few-shot comparison, use the **same three training examples** for both models.  
If you already have the Qwen `few_shot_examples.csv`, copy its three `train_line` values into `FIXED_TRAIN_LINES` in Cell 4.

Outputs:

- `/kaggle/working/prediction_banglallm_few_shot.csv`
- `/kaggle/working/result_banglallm_few_shot.csv`
- `/kaggle/working/few_shot_examples_banglallm.csv`

Metrics:

- Normalized Exact Match
- Token F1
- Fuzzy Match
- Corpus BLEU
- ROUGE-1
- ROUGE-2
- ROUGE-L
- METEOR
- BERTScore Precision
- BERTScore Recall
- BERTScore F1
- Token-limit Outputs


In [ ]:
# ============================================================
# CELL 1 — INSTALL DEPENDENCIES
# ============================================================

!pip install -q -U "transformers>=4.45,<5" accelerate bitsandbytes sentencepiece \
    sacrebleu rapidfuzz nltk "bert-score==0.3.13"


In [ ]:
# ============================================================
# CELL 2 — IMPORTS + PATHS + LOAD TEST AND TRAIN DATA
# ============================================================

import os
import re
import gc
import json
import random
import unicodedata
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from rapidfuzz import fuzz
from sacrebleu.metrics import BLEU
from nltk.translate.meteor_score import meteor_score
from bert_score import score as bert_score

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    set_seed,
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

set_seed(SEED)

MODEL_NAME = "BanglaLLM/bangla-llama-7b-instruct-v0.1"

TRAIN_JSONL = (
    "/kaggle/input/datasets/akra1234/government/"
    "government_chat_train.jsonl"
)

TEST_CSV = (
    "/kaggle/input/datasets/akra1234/government/"
    "merged_test_data.csv"
)

OUT_DIR = Path("/kaggle/working")

PRED_PATH = OUT_DIR / "prediction_banglallm_few_shot.csv"
PARTIAL_PATH = OUT_DIR / "prediction_banglallm_few_shot_partial.csv"
RESULT_PATH = OUT_DIR / "result_banglallm_few_shot.csv"
SHOTS_PATH = OUT_DIR / "few_shot_examples_banglallm.csv"

assert os.path.exists(TEST_CSV), (
    f"Test file not found: {TEST_CSV}"
)

assert os.path.exists(TRAIN_JSONL), (
    f"Training file not found: {TRAIN_JSONL}"
)

raw_df = pd.read_csv(TEST_CSV).fillna("")

print("Test rows:", len(raw_df))
print("Test columns:", raw_df.columns.tolist())


def choose_column(columns, candidates, required=True):
    lookup = {
        str(c).lower(): c
        for c in columns
    }

    for name in candidates:
        if name.lower() in lookup:
            return lookup[name.lower()]

    if required:
        raise ValueError(
            f"Could not find any of {candidates}. "
            f"Available columns: {list(columns)}"
        )

    return None


QUESTION_COL = choose_column(
    raw_df.columns,
    ["instruction", "question", "prompt", "query"],
)

GOLD_COL = choose_column(
    raw_df.columns,
    ["output", "gold", "reference", "answer", "target"],
)

INPUT_COL = choose_column(
    raw_df.columns,
    ["input"],
    required=False,
)

print("Question column:", QUESTION_COL)
print("Gold column:", GOLD_COL)
print("Optional input column:", INPUT_COL)

df = raw_df.copy()

df["question"] = (
    df[QUESTION_COL]
    .astype(str)
    .str.strip()
)

df["gold"] = (
    df[GOLD_COL]
    .astype(str)
    .str.strip()
)

if INPUT_COL is not None and INPUT_COL != QUESTION_COL:
    df["extra_input"] = (
        df[INPUT_COL]
        .astype(str)
        .str.strip()
    )
else:
    df["extra_input"] = ""

display(
    df[
        ["question", "extra_input", "gold"]
    ].head(3)
)


# ------------------------------------------------------------
# Robustly parse QA demonstrations from the training JSONL.
# Training "input" is kept separate so BanglaLLM can use its
# native optional ### Input: block.
# ------------------------------------------------------------

def _clean_text(x):
    if x is None:
        return ""

    if isinstance(x, (dict, list)):
        return json.dumps(
            x,
            ensure_ascii=False,
        )

    return str(x).strip()


def extract_train_example(obj):
    # OpenAI/ChatML-style messages
    messages = obj.get("messages")

    if isinstance(messages, list):
        user_text = ""
        assistant_text = ""

        for message in messages:
            if not isinstance(message, dict):
                continue

            role = str(
                message.get("role", "")
            ).lower().strip()

            content = _clean_text(
                message.get("content", "")
            )

            if role == "user" and not user_text:
                user_text = content

            elif (
                role == "assistant"
                and user_text
                and not assistant_text
            ):
                assistant_text = content
                break

        if user_text and assistant_text:
            return {
                "question": user_text,
                "extra_input": "",
                "answer": assistant_text,
            }

    # Alpaca-style / instruction-output schema
    question = ""

    for key in [
        "instruction",
        "question",
        "prompt",
        "query",
    ]:
        if key in obj and _clean_text(obj.get(key)):
            question = _clean_text(obj.get(key))
            break

    answer = ""

    for key in [
        "output",
        "answer",
        "response",
        "target",
        "gold",
        "reference",
    ]:
        if key in obj and _clean_text(obj.get(key)):
            answer = _clean_text(obj.get(key))
            break

    extra_input = _clean_text(
        obj.get("input", "")
    )

    if question and answer:
        return {
            "question": question,
            "extra_input": extra_input,
            "answer": answer,
        }

    # ShareGPT-style conversations
    conversations = obj.get("conversations")

    if isinstance(conversations, list):
        user_text = ""
        assistant_text = ""

        for message in conversations:
            if not isinstance(message, dict):
                continue

            role = str(
                message.get(
                    "from",
                    message.get("role", ""),
                )
            ).lower().strip()

            content = _clean_text(
                message.get(
                    "value",
                    message.get("content", ""),
                )
            )

            if (
                role in {"human", "user"}
                and not user_text
            ):
                user_text = content

            elif (
                role in {"gpt", "assistant", "bot"}
                and user_text
                and not assistant_text
            ):
                assistant_text = content
                break

        if user_text and assistant_text:
            return {
                "question": user_text,
                "extra_input": "",
                "answer": assistant_text,
            }

    return None


train_records = []

with open(
    TRAIN_JSONL,
    "r",
    encoding="utf-8",
) as file:
    for line_no, line in enumerate(
        file,
        start=1,
    ):
        line = line.strip()

        if not line:
            continue

        obj = json.loads(line)

        example = extract_train_example(obj)

        if example is None:
            continue

        train_records.append({
            "train_line": line_no,
            "question": example["question"].strip(),
            "extra_input": example["extra_input"].strip(),
            "answer": example["answer"].strip(),
        })


train_df = pd.DataFrame(train_records)

if train_df.empty:
    raise ValueError(
        "No usable question-answer pairs were parsed "
        "from the training JSONL."
    )

train_df = train_df[
    train_df["question"]
    .astype(str)
    .str.strip()
    .ne("")
    &
    train_df["answer"]
    .astype(str)
    .str.strip()
    .ne("")
].copy()

print(
    "Parsed train QA pairs:",
    len(train_df),
)


# ------------------------------------------------------------
# Remove exact train/test QUESTION overlap before selecting shots.
# This avoids leaking a test answer through a demonstration.
# ------------------------------------------------------------

BN_TO_EN_OVERLAP = str.maketrans(
    "০১২৩৪৫৬৭৮৯",
    "0123456789",
)


def normalize_question_for_overlap(text):
    text = unicodedata.normalize(
        "NFKC",
        str(text),
    )

    text = (
        text.translate(BN_TO_EN_OVERLAP)
        .lower()
    )

    text = re.sub(
        r"[^\u0980-\u09FFA-Za-z0-9]+",
        " ",
        text,
    )

    return re.sub(
        r"\s+",
        " ",
        text,
    ).strip()


test_question_norms = set(
    df["question"].map(
        normalize_question_for_overlap
    )
)

train_df["question_norm"] = (
    train_df["question"]
    .map(normalize_question_for_overlap)
)

before_overlap_filter = len(train_df)

train_pool_df = train_df[
    ~train_df["question_norm"].isin(
        test_question_norms
    )
].copy()

removed_overlap_rows = (
    before_overlap_filter
    - len(train_pool_df)
)

# Avoid duplicate demonstrations with the same question.
train_pool_df = (
    train_pool_df
    .drop_duplicates(
        subset=["question_norm"],
        keep="first",
    )
    .reset_index(drop=True)
)

print(
    "Train/test exact-overlap rows excluded:",
    removed_overlap_rows,
)

print(
    "Unique non-overlapping train pool:",
    len(train_pool_df),
)

if len(train_pool_df) < 3:
    raise ValueError(
        "Fewer than 3 non-overlapping training "
        "examples remain."
    )


In [ ]:
# ============================================================
# CELL 3 — LOAD BANGLALLM 7B INSTRUCT IN 4-BIT
# ============================================================

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is strongly recommended. "
        "In Kaggle: Settings -> Accelerator -> GPU."
    )

compute_dtype = torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)

if tokenizer.pad_token_id is None:
    if tokenizer.eos_token_id is None:
        raise ValueError(
            "Tokenizer has neither a PAD nor EOS token."
        )

    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=compute_dtype,
)

model.eval()

MODEL_CONTEXT_WINDOW = int(
    getattr(
        model.config,
        "max_position_embeddings",
        4096,
    )
)

print("Loaded:", MODEL_NAME)
print("Device:", next(model.parameters()).device)
print("Context window:", MODEL_CONTEXT_WINDOW)
print("Tokenizer EOS:", tokenizer.eos_token_id)
print(
    "Model generation EOS:",
    model.generation_config.eos_token_id,
)


In [ ]:
# ============================================================
# CELL 4 — SELECT 3 FIXED FEW-SHOT DEMONSTRATIONS
# ============================================================

NUM_SHOTS = 3

# Keeps unusually long examples from consuming most of the
# 4096-token BanglaLLM context window.
MAX_DEMO_PAIR_TOKENS = 512

# ------------------------------------------------------------
# FAIRNESS OPTION
# ------------------------------------------------------------
# For strict Qwen-vs-BanglaLLM few-shot comparison, put the
# three Qwen demonstration train_line values here.
#
# Example:
# FIXED_TRAIN_LINES = [123, 456, 789]
#
# Leave as None to perform reproducible seeded selection.
FIXED_TRAIN_LINES = None


def format_instruction_block(
    question,
    extra_input="",
    answer=None,
):
    """
    Format one BanglaLLM Alpaca-style example.
    """
    question = str(question).strip()
    extra_input = str(extra_input).strip()

    parts = [
        f"### Instruction:\n{question}"
    ]

    if extra_input:
        parts.append(
            f"### Input:\n{extra_input}"
        )

    if answer is None:
        parts.append(
            "### Response:\n"
        )
    else:
        parts.append(
            "### Response:\n"
            + str(answer).strip()
        )

    return "\n\n".join(parts)


def demonstration_token_length(row):
    demo_text = format_instruction_block(
        row["question"],
        row["extra_input"],
        row["answer"],
    )

    return len(
        tokenizer(
            demo_text,
            add_special_tokens=False,
            truncation=False,
        )["input_ids"]
    )


if FIXED_TRAIN_LINES is not None:
    fixed_lines = [
        int(x)
        for x in FIXED_TRAIN_LINES
    ]

    if len(fixed_lines) != NUM_SHOTS:
        raise ValueError(
            f"FIXED_TRAIN_LINES must contain exactly "
            f"{NUM_SHOTS} train line numbers."
        )

    if len(set(fixed_lines)) != NUM_SHOTS:
        raise ValueError(
            "FIXED_TRAIN_LINES contains duplicates."
        )

    selected = train_pool_df[
        train_pool_df["train_line"].isin(
            fixed_lines
        )
    ].copy()

    missing = (
        set(fixed_lines)
        - set(
            selected["train_line"]
            .astype(int)
            .tolist()
        )
    )

    if missing:
        raise ValueError(
            "These requested train lines are not available "
            "in the safe non-overlapping training pool: "
            f"{sorted(missing)}"
        )

    # Reorder exactly as FIXED_TRAIN_LINES.
    selected["_order"] = selected[
        "train_line"
    ].map({
        line: idx
        for idx, line in enumerate(fixed_lines)
    })

    selected = (
        selected
        .sort_values("_order")
        .drop(columns=["_order"])
        .reset_index(drop=True)
    )

    selected_rows = [
        row
        for _, row in selected.iterrows()
    ]

else:
    shuffled_train = (
        train_pool_df
        .sample(
            frac=1.0,
            random_state=SEED,
        )
        .reset_index(drop=True)
    )

    selected_rows = []

    for _, row in shuffled_train.iterrows():
        pair_length = (
            demonstration_token_length(row)
        )

        if pair_length <= MAX_DEMO_PAIR_TOKENS:
            selected_rows.append(row)

        if len(selected_rows) == NUM_SHOTS:
            break


if len(selected_rows) < NUM_SHOTS:
    raise ValueError(
        f"Could select only {len(selected_rows)} "
        f"demonstrations under "
        f"MAX_DEMO_PAIR_TOKENS="
        f"{MAX_DEMO_PAIR_TOKENS}."
    )


shots_df = (
    pd.DataFrame(selected_rows)
    .reset_index(drop=True)
)

shots_df.insert(
    0,
    "shot_id",
    range(1, NUM_SHOTS + 1),
)

shots_df["demo_tokens"] = [
    demonstration_token_length(row)
    for _, row in shots_df.iterrows()
]

shots_df[
    [
        "shot_id",
        "train_line",
        "question",
        "extra_input",
        "answer",
        "demo_tokens",
    ]
].to_csv(
    SHOTS_PATH,
    index=False,
    encoding="utf-8-sig",
)

print(
    f"Selected {NUM_SHOTS} fixed demonstrations."
)

print(
    "Selected train lines:",
    shots_df["train_line"]
    .astype(int)
    .tolist(),
)

print(
    "Saved demonstration audit:",
    SHOTS_PATH,
)

for _, row in shots_df.iterrows():
    print("\n" + "=" * 70)

    print(
        f"SHOT {int(row['shot_id'])} "
        f"| train line "
        f"{int(row['train_line'])} "
        f"| tokens "
        f"{int(row['demo_tokens'])}"
    )

    print("- Question:")
    print(row["question"])

    if str(row["extra_input"]).strip():
        print("- Input:")
        print(row["extra_input"])

    print("- Answer:")
    print(row["answer"])


display(
    shots_df[
        [
            "shot_id",
            "train_line",
            "question",
            "extra_input",
            "answer",
            "demo_tokens",
        ]
    ]
)


In [ ]:
# ============================================================
# CELL 5 — BANGLALLM 3-SHOT GENERATION
# ============================================================

SYSTEM_PROMPT = (
    "আপনি বাংলাদেশের সরকারি সেবা সম্পর্কিত প্রশ্নের সহায়ক। "
    "নিচের উদাহরণগুলোর উত্তর দেওয়ার ধরন অনুসরণ করুন। "
    "ব্যবহারকারীর নতুন প্রশ্নের উত্তর বাংলায় দিন। "
    "উত্তরটি সংক্ষিপ্ত, সরাসরি ও তথ্যভিত্তিক রাখুন। "
    "তথ্য না জানলে বানিয়ে বলবেন না। "
    "উদাহরণের প্রশ্নের উত্তর কপি করবেন না; "
    "নতুন প্রশ্নের জন্য প্রাসঙ্গিক উত্তর দিন। "
    "কোনো reference answer, dataset, training example "
    "বা evaluation-এর কথা উল্লেখ করবেন না।"
)

BATCH_SIZE = 4

# IMPORTANT:
# Keep these generation settings the same as the BanglaLLM
# zero-shot notebook so the demonstrations are the main changed variable.
MAX_NEW_TOKENS = 256

DO_SAMPLE = True
TEMPERATURE = 0.6
TOP_P = 0.9
TOP_K = 50

CONTEXT_SAFETY_MARGIN = 16

MAX_PROMPT_TOKENS = (
    MODEL_CONTEXT_WINDOW
    - MAX_NEW_TOKENS
    - CONTEXT_SAFETY_MARGIN
)

if MAX_PROMPT_TOKENS <= 0:
    raise ValueError(
        "Invalid prompt/generation token budget."
    )

# True = delete stale outputs and regenerate every test row.
FORCE_REGENERATE = True


def make_few_shot_prompt(
    question,
    extra_input="",
):
    """
    BanglaLLM 3-shot prompt:

    system prompt

    ### Instruction:
    demo 1 question
    ### Response:
    demo 1 answer

    ... repeated for 3 demos ...

    ### Instruction:
    test question
    ### Response:
    """
    blocks = [
        SYSTEM_PROMPT
    ]

    for _, shot in shots_df.iterrows():
        blocks.append(
            format_instruction_block(
                shot["question"],
                shot["extra_input"],
                shot["answer"],
            )
        )

    blocks.append(
        format_instruction_block(
            question,
            extra_input,
            answer=None,
        )
    )

    return "\n\n".join(blocks)


def _get_eos_token_ids():
    eos = model.generation_config.eos_token_id

    if eos is None:
        eos = tokenizer.eos_token_id

    if eos is None:
        raise ValueError(
            "No EOS token id is configured."
        )

    if isinstance(eos, int):
        eos_ids = [eos]
    else:
        eos_ids = list(eos)

    if tokenizer.eos_token_id is not None:
        eos_ids.append(
            int(tokenizer.eos_token_id)
        )

    return list(
        dict.fromkeys(
            int(x)
            for x in eos_ids
        )
    )


EOS_TOKEN_IDS = _get_eos_token_ids()

GEN_EOS = (
    EOS_TOKEN_IDS[0]
    if len(EOS_TOKEN_IDS) == 1
    else EOS_TOKEN_IDS
)

EOS_TOKEN_ID_SET = set(
    EOS_TOKEN_IDS
)

print("EOS token ids:", EOS_TOKEN_IDS)
print(
    "Model context window:",
    MODEL_CONTEXT_WINDOW,
)
print(
    "Maximum allowed few-shot prompt tokens:",
    MAX_PROMPT_TOKENS,
)
print(
    "Max new tokens:",
    MAX_NEW_TOKENS,
)
print("Sampling:", DO_SAMPLE)
print(
    "temperature/top_p/top_k:",
    TEMPERATURE,
    TOP_P,
    TOP_K,
)


# ------------------------------------------------------------
# Preview one full prompt BEFORE generation.
# This lets you verify the demonstrations and final question.
# ------------------------------------------------------------

preview_prompt = make_few_shot_prompt(
    df.iloc[0]["question"],
    df.iloc[0]["extra_input"],
)

preview_tokens = len(
    tokenizer(
        preview_prompt,
        add_special_tokens=True,
        truncation=False,
    )["input_ids"]
)

print(
    "\nFirst few-shot prompt tokens:",
    preview_tokens,
)

print("\n" + "=" * 70)
print("FEW-SHOT PROMPT PREVIEW")
print("=" * 70)
print(preview_prompt[:5000])
print("=" * 70)


def generate_batch(
    questions,
    extra_inputs,
):
    prompts = [
        make_few_shot_prompt(
            question,
            extra_input,
        )
        for question, extra_input in zip(
            questions,
            extra_inputs,
        )
    ]

    prompt_token_lengths = [
        len(
            tokenizer(
                prompt,
                add_special_tokens=True,
                truncation=False,
            )["input_ids"]
        )
        for prompt in prompts
    ]

    longest_prompt = max(
        prompt_token_lengths
    )

    if longest_prompt > MAX_PROMPT_TOKENS:
        raise RuntimeError(
            "A few-shot prompt is too long for the "
            "BanglaLLM context window. "
            f"Longest prompt={longest_prompt} tokens, "
            f"allowed={MAX_PROMPT_TOKENS}. "
            "The notebook refuses to silently truncate "
            "the test question. Reduce demonstration length "
            "or use shorter fixed shots."
        )

    batch = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=False,
    )

    input_device = (
        next(model.parameters()).device
    )

    batch = {
        key: value.to(input_device)
        for key, value in batch.items()
    }

    input_width = (
        batch["input_ids"].shape[1]
    )

    with torch.inference_mode():
        generated = model.generate(
            **batch,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=DO_SAMPLE,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            top_k=TOP_K,
            num_beams=1,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=GEN_EOS,
        )

    new_tokens = generated[
        :,
        input_width:,
    ]

    token_rows = (
        new_tokens
        .detach()
        .cpu()
        .tolist()
    )

    answers = []
    generated_lengths = []
    eos_flags = []
    token_limit_flags = []

    for token_ids in token_rows:
        first_eos_position = next(
            (
                idx
                for idx, token_id
                in enumerate(token_ids)
                if token_id in EOS_TOKEN_ID_SET
            ),
            None,
        )

        if first_eos_position is not None:
            effective_ids = token_ids[
                :first_eos_position + 1
            ]

            generated_length = (
                first_eos_position + 1
            )

            finished_with_eos = True
            hit_token_limit = False

        else:
            effective_ids = token_ids[
                :MAX_NEW_TOKENS
            ]

            generated_length = min(
                len(token_ids),
                MAX_NEW_TOKENS,
            )

            finished_with_eos = False

            hit_token_limit = (
                generated_length
                >= MAX_NEW_TOKENS
            )

        answer = tokenizer.decode(
            effective_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        ).strip()

        answers.append(answer)

        generated_lengths.append(
            int(generated_length)
        )

        eos_flags.append(
            bool(finished_with_eos)
        )

        token_limit_flags.append(
            bool(hit_token_limit)
        )

    return (
        answers,
        generated_lengths,
        eos_flags,
        token_limit_flags,
        prompt_token_lengths,
    )


def save_partial(frame):
    (
        frame
        .sort_values("row_index")
        .to_csv(
            PARTIAL_PATH,
            index=False,
            encoding="utf-8-sig",
        )
    )


# ------------------------------------------------------------
# Clean start / optional resume
# ------------------------------------------------------------

if FORCE_REGENERATE:
    for old_path in [
        PARTIAL_PATH,
        PRED_PATH,
        RESULT_PATH,
    ]:
        if old_path.exists():
            old_path.unlink()

            print(
                "Removed old output:",
                old_path,
            )


predictions = []

if (
    not FORCE_REGENERATE
    and PARTIAL_PATH.exists()
):
    partial = (
        pd.read_csv(PARTIAL_PATH)
        .fillna("")
    )

    required_cols = {
        "row_index",
        "question",
        "gold",
        "prediction",
        "generated_tokens",
        "finished_with_eos",
        "hit_token_limit",
        "truncated",
        "max_new_tokens_used",
        "generation_attempts",
        "num_shots",
        "shot_train_lines",
        "prompt_tokens",
    }

    expected_indices = list(
        range(len(partial))
    )

    actual_indices = (
        pd.to_numeric(
            partial.get(
                "row_index",
                pd.Series(dtype=int),
            ),
            errors="coerce",
        )
        .tolist()
    )

    if (
        len(partial) <= len(df)
        and required_cols.issubset(
            partial.columns
        )
        and actual_indices == expected_indices
    ):
        predictions = (
            partial.to_dict("records")
        )

        print(
            "Resuming from",
            len(predictions),
            "completed rows.",
        )

    else:
        print(
            "Ignoring incompatible partial file "
            "and starting fresh."
        )


# ------------------------------------------------------------
# Generate all test rows once
# ------------------------------------------------------------

start = len(predictions)

shot_train_lines = "|".join(
    shots_df["train_line"]
    .astype(int)
    .astype(str)
    .tolist()
)

for i in tqdm(
    range(
        start,
        len(df),
        BATCH_SIZE,
    ),
    desc=(
        f"BanglaLLM {NUM_SHOTS}-shot generation "
        f"({MAX_NEW_TOKENS} max tokens)"
    ),
):
    end = min(
        i + BATCH_SIZE,
        len(df),
    )

    questions = (
        df.iloc[i:end]["question"]
        .tolist()
    )

    extra_inputs = (
        df.iloc[i:end]["extra_input"]
        .tolist()
    )

    # Sampling is reproducible at the batch level.
    set_seed(SEED + i)

    (
        answers,
        generated_lengths,
        eos_flags,
        token_limit_flags,
        prompt_token_lengths,
    ) = generate_batch(
        questions,
        extra_inputs,
    )

    for local_idx, (
        answer,
        generated_length,
        eos_ok,
        hit_limit,
        prompt_length,
    ) in enumerate(
        zip(
            answers,
            generated_lengths,
            eos_flags,
            token_limit_flags,
            prompt_token_lengths,
        )
    ):
        row_idx = i + local_idx
        row = df.iloc[row_idx]

        record = {
            "row_index": row_idx,
            "question": row["question"],
            "gold": row["gold"],
            "prediction": answer,
            "generated_tokens": int(
                generated_length
            ),
            "finished_with_eos": bool(
                eos_ok
            ),
            "hit_token_limit": bool(
                hit_limit
            ),
            # Compatibility with previous notebooks.
            "truncated": bool(
                hit_limit
            ),
            "max_new_tokens_used": int(
                MAX_NEW_TOKENS
            ),
            "generation_attempts": 1,
            "num_shots": NUM_SHOTS,
            "shot_train_lines": (
                shot_train_lines
            ),
            "prompt_tokens": int(
                prompt_length
            ),
        }

        for col in [
            "id",
            "domain",
            "topic",
            "question_type",
            "source_url",
            "split",
        ]:
            if col in df.columns:
                record[col] = row[col]

        predictions.append(record)

    save_partial(
        pd.DataFrame(predictions)
    )


pred_df = (
    pd.DataFrame(predictions)
    .sort_values("row_index")
    .reset_index(drop=True)
)

assert len(pred_df) == len(df), (
    f"Generated {len(pred_df)} predictions "
    f"for {len(df)} test rows."
)

pred_df.to_csv(
    PRED_PATH,
    index=False,
    encoding="utf-8-sig",
)

token_limit_count = int(
    pred_df["hit_token_limit"]
    .astype(bool)
    .sum()
)

eos_count = int(
    pred_df["finished_with_eos"]
    .astype(bool)
    .sum()
)

empty_output_count = int(
    pred_df["prediction"]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

print(
    "\nGeneration complete:",
    len(pred_df),
)

print(
    "Finished with EOS:",
    eos_count,
)

print(
    "Hit token limit:",
    token_limit_count,
)

print(
    "Empty outputs:",
    empty_output_count,
)

print(
    "Average prompt tokens:",
    round(
        pd.to_numeric(
            pred_df["prompt_tokens"],
            errors="coerce",
        ).mean(),
        2,
    ),
)

print(
    "Saved predictions:",
    PRED_PATH,
)

# IMPORTANT:
# Do not automatically give token-limit rows a bigger budget.
# The same 256-token generation protocol is retained from the
# BanglaLLM zero-shot experiment, making zero-shot vs few-shot
# comparison cleaner.
display(
    pred_df.head(3)
)


In [ ]:
# ============================================================
# CELL 6 — FREE BANGLALLM GPU MEMORY BEFORE BERTSCORE
# ============================================================

del model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(
    "BanglaLLM model removed from GPU memory."
)


In [ ]:
# ============================================================
# CELL 7 — METRIC FUNCTIONS
# Keeps the same normalization / Token F1 / ROUGE definitions
# as the previous evaluation pipeline.
# ============================================================

BN_TO_EN = str.maketrans(
    "০১২৩৪৫৬৭৮৯",
    "0123456789"
)


def normalize(text):

    text = unicodedata.normalize(
        "NFKC",
        str(text)
    )

    text = text.translate(
        BN_TO_EN
    ).lower()

    text = re.sub(
        r"[^\u0980-\u09FFA-Za-z0-9]+",
        " ",
        text
    )

    return re.sub(
        r"\s+",
        " ",
        text
    ).strip()


def tokens(text):
    return normalize(text).split()


# ---------------- Normalized Exact Match ----------------

def normalized_exact_match(pred, gold):

    return float(
        normalize(pred)
        ==
        normalize(gold)
    )


# ---------------- Token F1 ----------------

def token_f1(pred, gold):

    p = tokens(pred)
    g = tokens(gold)

    if not p and not g:
        return 1.0

    if not p or not g:
        return 0.0

    overlap = sum(
        (
            Counter(p)
            &
            Counter(g)
        ).values()
    )

    if overlap == 0:
        return 0.0

    precision = overlap / len(p)
    recall = overlap / len(g)

    return (
        2 * precision * recall
        /
        (precision + recall)
    )


# ---------------- ROUGE-N F1 ----------------

def rouge_n(pred, gold, n):

    p = tokens(pred)
    g = tokens(gold)

    if len(p) < n or len(g) < n:
        return 0.0

    pg = Counter(
        tuple(p[i:i+n])
        for i in range(len(p)-n+1)
    )

    gg = Counter(
        tuple(g[i:i+n])
        for i in range(len(g)-n+1)
    )

    overlap = sum(
        (pg & gg).values()
    )

    if overlap == 0:
        return 0.0

    precision = overlap / sum(pg.values())
    recall = overlap / sum(gg.values())

    return (
        2 * precision * recall
        /
        (precision + recall)
    )


# ---------------- ROUGE-L F1 ----------------

def rouge_l(pred, gold):

    p = tokens(pred)
    g = tokens(gold)

    if not p and not g:
        return 1.0

    if not p or not g:
        return 0.0

    dp = [0] * (len(g) + 1)

    for x in p:

        new = [0]

        for j, y in enumerate(g, 1):

            if x == y:
                new.append(dp[j-1] + 1)

            else:
                new.append(
                    max(dp[j], new[-1])
                )

        dp = new

    lcs = dp[-1]

    precision = lcs / len(p)
    recall = lcs / len(g)

    if precision + recall == 0:
        return 0.0

    return (
        2 * precision * recall
        /
        (precision + recall)
    )


# ---------------- Bengali-safe METEOR ----------------
# NLTK's default METEOR uses English Porter stemming + English WordNet.
# For Bangla evaluation, disable those English-only lexical resources while
# retaining METEOR's exact-token alignment and fragmentation penalty.

class IdentityStemmer:
    def stem(self, word):
        return word


class EmptyWordNet:
    def synsets(self, word):
        return []


IDENTITY_STEMMER = IdentityStemmer()
EMPTY_WORDNET = EmptyWordNet()


def meteor_bn(pred, gold):

    p = tokens(pred)
    g = tokens(gold)

    if not p and not g:
        return 1.0

    if not p or not g:
        return 0.0

    return meteor_score(
        [g],
        p,
        stemmer=IDENTITY_STEMMER,
        wordnet=EMPTY_WORDNET,
    )


In [ ]:
# ============================================================
# CELL 8 — COMPUTE ROW-LEVEL METRICS
# ============================================================

eval_df = pd.read_csv(PRED_PATH).fillna("")

# For this benchmark, evaluate every generated prediction.
# Token-limit failures are retained as model behavior rather than
# silently excluded or regenerated with a larger budget.
n_token_limit_for_eval = 0

if "hit_token_limit" in eval_df.columns:
    n_token_limit_for_eval = int(
        eval_df["hit_token_limit"]
        .astype(str)
        .str.lower()
        .eq("true")
        .sum()
    )

print(
    f"Evaluating all {len(eval_df)} "
    "BanglaLLM 3-shot predictions..."
)

print(
    "Token-limit outputs included in evaluation:",
    n_token_limit_for_eval,
)

eval_df["Normalized Exact Match"] = [
    normalized_exact_match(p, g)
    for p, g in zip(
        eval_df["prediction"],
        eval_df["gold"]
    )
]

eval_df["Token F1"] = [
    token_f1(p, g)
    for p, g in zip(
        eval_df["prediction"],
        eval_df["gold"]
    )
]

eval_df["Fuzzy Match"] = [
    fuzz.token_set_ratio(
        normalize(p),
        normalize(g)
    ) / 100
    for p, g in zip(
        eval_df["prediction"],
        eval_df["gold"]
    )
]

eval_df["ROUGE-1"] = [
    rouge_n(p, g, 1)
    for p, g in zip(
        eval_df["prediction"],
        eval_df["gold"]
    )
]

eval_df["ROUGE-2"] = [
    rouge_n(p, g, 2)
    for p, g in zip(
        eval_df["prediction"],
        eval_df["gold"]
    )
]

eval_df["ROUGE-L"] = [
    rouge_l(p, g)
    for p, g in zip(
        eval_df["prediction"],
        eval_df["gold"]
    )
]

eval_df["METEOR"] = [
    meteor_bn(p, g)
    for p, g in tqdm(
        zip(
            eval_df["prediction"],
            eval_df["gold"]
        ),
        total=len(eval_df),
        desc="METEOR"
    )
]


# ============================================================
# CORPUS BLEU
# ============================================================

bleu = BLEU(
    tokenize="none",
    smooth_method="exp",
    effective_order=True
)

pred_texts = [
    " ".join(tokens(x))
    for x in eval_df["prediction"]
]

gold_texts = [
    " ".join(tokens(x))
    for x in eval_df["gold"]
]

corpus_bleu = (
    bleu.corpus_score(
        pred_texts,
        [gold_texts]
    ).score
    / 100
)

print("Corpus BLEU:", corpus_bleu)


In [ ]:
# ============================================================
# CELL 9 — BERTSCORE
# model: bert-base-multilingual-cased
# ============================================================

print("Calculating multilingual BERTScore...")

bert_device = "cuda" if torch.cuda.is_available() else "cpu"
bert_batch_size = 8 if torch.cuda.is_available() else 4

P, R, F1 = bert_score(
    eval_df["prediction"].astype(str).tolist(),
    eval_df["gold"].astype(str).tolist(),
    model_type="bert-base-multilingual-cased",
    batch_size=bert_batch_size,
    device=bert_device,
    idf=False,
    rescale_with_baseline=False,
    verbose=True
)

eval_df["BERTScore Precision"] = P.cpu().numpy()
eval_df["BERTScore Recall"] = R.cpu().numpy()
eval_df["BERTScore F1"] = F1.cpu().numpy()

print("BERTScore complete.")


In [ ]:
# ============================================================
# CELL 10 — FINAL RESULT + SAVE BANGLALLM FEW-SHOT OUTPUTS
# ============================================================

token_limit_count = int(
    eval_df["hit_token_limit"]
    .astype(str)
    .str.lower()
    .eq("true")
    .sum()
)

eos_count = int(
    eval_df["finished_with_eos"]
    .astype(str)
    .str.lower()
    .eq("true")
    .sum()
)

empty_output_count = int(
    eval_df["prediction"]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

avg_generated_tokens = (
    pd.to_numeric(
        eval_df["generated_tokens"],
        errors="coerce",
    ).mean()
    if "generated_tokens" in eval_df.columns
    else np.nan
)

max_generated_tokens = (
    pd.to_numeric(
        eval_df["generated_tokens"],
        errors="coerce",
    ).max()
    if "generated_tokens" in eval_df.columns
    else np.nan
)

avg_prompt_tokens = (
    pd.to_numeric(
        eval_df["prompt_tokens"],
        errors="coerce",
    ).mean()
    if "prompt_tokens" in eval_df.columns
    else np.nan
)

result = pd.DataFrame({
    "metric": [
        "Normalized Exact Match",
        "Token F1",
        "Fuzzy Match",
        "Corpus BLEU",
        "ROUGE-1",
        "ROUGE-2",
        "ROUGE-L",
        "METEOR",
        "BERTScore Precision",
        "BERTScore Recall",
        "BERTScore F1",
        "Token-limit Outputs",
        "Finished-with-EOS Outputs",
        "Empty Outputs",
        "Average Prompt Tokens",
        "Average Generated Tokens",
        "Maximum Generated Tokens",
    ],
    "score": [
        eval_df[
            "Normalized Exact Match"
        ].mean(),
        eval_df["Token F1"].mean(),
        eval_df["Fuzzy Match"].mean(),
        corpus_bleu,
        eval_df["ROUGE-1"].mean(),
        eval_df["ROUGE-2"].mean(),
        eval_df["ROUGE-L"].mean(),
        eval_df["METEOR"].mean(),
        eval_df[
            "BERTScore Precision"
        ].mean(),
        eval_df[
            "BERTScore Recall"
        ].mean(),
        eval_df[
            "BERTScore F1"
        ].mean(),
        token_limit_count,
        eos_count,
        empty_output_count,
        avg_prompt_tokens,
        avg_generated_tokens,
        max_generated_tokens,
    ],
})

# Save row-level predictions + metrics.
eval_df.to_csv(
    PRED_PATH,
    index=False,
    encoding="utf-8-sig",
)

# Save aggregate metrics.
result.to_csv(
    RESULT_PATH,
    index=False,
    encoding="utf-8-sig",
)

display(result)

print(
    f"\nExperiment: BanglaLLM "
    f"{NUM_SHOTS}-shot, seed={SEED}"
)

print(
    "Selected shot train lines:",
    shots_df["train_line"]
    .astype(int)
    .tolist(),
)

print("\nSaved:")
print(PRED_PATH)
print(RESULT_PATH)
print(SHOTS_PATH)

print("\nValidation:")
print(
    "Rows evaluated:",
    len(eval_df),
)
print(
    "Token-limit outputs:",
    token_limit_count,
)
print(
    "Finished with EOS:",
    eos_count,
)
print(
    "Empty outputs:",
    empty_output_count,
)

print("\nNOTE:")
print(
    "/kaggle/input is read-only. "
    "Kaggle outputs must be written "
    "under /kaggle/working."
)
